# Method B: compute the criteria directly

**Paper output being replaced:** Table 10. Each industry gets a score (low = more polluting) and is ranked within its industrial area.
`score = Σ criterion_weight(C1..C10) × local_weight(industry, criterion)`. The experts produced the local weights (Table 6 ratings, then fuzzy AHP).

Per-industry summary (`wqml.summarise`): one row per industry, one column per paper criterion, all computed from lab data.

| Criterion | Summary value | Worse when |
|---|---|---|
| C1 Effluent discharge | mean flow (MLD) | higher |
| C2 ETP/CETP efficiency | mean (inlet − outlet) / inlet | lower |
| C3 / C4 Critical parameters, pre- / post-monsoon | pollutants over the CPCB limit per sampling event | higher |
| C5 Effluent type | industry type → paper's 1–9 scale (Table 4) | lower |
| C6 / C7 River impact, pre- / post-monsoon | downstream − upstream, percentile within pollutant | higher |
| C8 / C9 Groundwater impact, pre- / post-monsoon | near well − control well, percentile within pollutant | higher |
| C10 Public health | Σ toxicity × amount over the limit, per event | higher |

**Method B:** rank industries within their area on each criterion, scale ranks to sum to 1 (the shape of the paper's local weights), then add with fixed criterion weights. No expert rates an industry.

**Weights.** Main result: weights chosen **without experts** (equal, or CRITIC from the data). The paper's expert weights are shown only as a reference: with them the method copies part of the experts' judgment, so their agreement says little.

**What counts as bad:**
- **Legal limits** (C3, C4, C10): CPCB standards; pH outside 5.5–9.0. A non-detect counts as a pass.
- **Physical direction** (C1, C2, C6–C9): more discharge, lower treatment efficiency or a larger rise from upstream to downstream is worse, relative to the area (the paper's scores are relative within areas too).
- **Value choices set once** (C5 scale, toxicity factors): placeholders in `wqml`, to be confirmed by experts.

**Missing values.** Measured ranks are stretched onto the full scale so their average is neutral; missing = neutral; a criterion measured for fewer than 2 industries in an area is neutral for that area (`wqml.local_weights`).

**Agreement** is measured *within* each area (the paper ranks only within areas). Agreement with the experts is not proof of a better assessment: it only shows how close the computed ranking is to theirs.

In [ ]:
import numpy as np
import pandas as pd
import wqml
from wqml import CRITERIA, WORSE_IF_HIGH

## 1. Data and per-industry summary
`n_tested` = parameters tested at the outlet; C3/C4/C10 can grow with a wider panel (see section 5).

In [ ]:
raw, SOURCE = wqml.load()  # your data, else the paper's source report (data/uppcb/), else dummy
DUMMY = SOURCE == "dummy"
print({"user": "Your data.", "uppcb": "Paper's source data: UPPCB/NIH 2013 report, 54 stations, May and Nov 2012.",
       "dummy": "DUMMY data: results show the code works, nothing about the real industries."}[SOURCE])
df = wqml.prepare(raw)
summary = wqml.summarise(df)
labels = wqml.paper_labels()
summary.round(3)

## 2. Weights chosen once
*equal*: same weight for every criterion with data for at least 10 industries. *CRITIC*: weight grows with a criterion's spread and its disagreement with the others (missing filled with the median, then scaled 0–1). *paper*: the experts' weights (reference only).

In [ ]:
used = [c for c in CRITERIA if summary[c].notna().sum() >= 10]
G = pd.DataFrame({c: (-summary[c] if WORSE_IF_HIGH[c] else summary[c]) for c in used})
G = G.fillna(G.median())
G = (G - G.min()) / (G.max() - G.min())
critic = G.std() * (1 - G.corr()).sum()
WEIGHTS = {
    "equal": pd.Series({c: (1 / len(used) if c in used else 0.0) for c in CRITERIA}),
    "CRITIC": pd.Series({c: (critic[c] / critic.sum() if c in used else 0.0) for c in CRITERIA}),
    "paper (reference)": pd.read_csv(wqml.PAPER / "criteria.csv").set_index("criterion")["weight"],
}
print("criteria with data:", used)
pd.DataFrame(WEIGHTS).round(3)

## 3. Compute and compare with the paper's ranking

In [ ]:
local = wqml.local_weights(summary)


def method_b(w):
    score = (local * w[CRITERIA]).sum(axis=1)
    return score / score.groupby(summary["industrial_area"]).transform("mean")  # 1 = area average


def explain(source, w):
    """Each criterion's weighted contribution vs a neutral (average) industry in the same area.
    Negative = pushed it towards 'more polluting'."""
    n = (summary["industrial_area"] == summary.loc[source, "industrial_area"]).sum()
    return pd.DataFrame({"measured": summary.loc[source, CRITERIA],
                         "vs_neutral": ((local.loc[source] - 1 / n) * w[CRITERIA]).round(4)})


t = summary[["industrial_area"]].join(labels)
for name, w in WEIGHTS.items():
    t[name] = method_b(w)
pd.DataFrame({name: wqml.agreement(t, name) for name in WEIGHTS}).T

Ranking with **equal** weights (rank 1 = most polluting in its area) next to the paper's.

In [ ]:
t["B_rank"] = t.groupby("industrial_area")["equal"].rank(method="min").astype(int)
t["fuzzy_rank"] = t.groupby("industrial_area")["fuzzy_weight"].rank(method="min").astype(int)
t[["industrial_area", "fuzzy_rank", "B_rank", "fuzzy_weight", "equal"]].sort_values(["industrial_area", "fuzzy_rank"]).round(3)

In [ ]:
worst = t.groupby("industrial_area")["equal"].idxmin()
print("Most polluting per area (B, equal weights):", worst.to_dict())
explain(worst.iloc[0], WEIGHTS["equal"])

## 4. How much do the weights matter?
2,000 random weight sets over the same criteria (Dirichlet, seed 0).

In [ ]:
rng = np.random.default_rng(0)
rand = [wqml.agreement(t.assign(r=(local[used] * rng.dirichlet(np.ones(len(used)))).sum(axis=1)), "r")["spearman_within_area"]
        for _ in range(2000)]
print(f"within-area Spearman over random weights: median {np.median(rand):.3f}, 5-95% [{np.quantile(rand, .05):.3f}, {np.quantile(rand, .95):.3f}]")
pd.Series({c: wqml.agreement(t.assign(one=local[c]), "one")["spearman_within_area"] for c in used}, name="one criterion alone").round(3)

## 5. Sensitivity: size of the tested panel
Outlet panels range from 1 to 17 parameters, so a count of failures (C3/C4) or a sum of excess (C10) can grow just because more was tested. Variant: divide by the number of parameters tested in the same sampling event.

In [ ]:
per_test = wqml.summarise(df, how="mean")  # failures / excess per parameter tested in the same event
local_pt = wqml.local_weights(per_test)
score_pt = (local_pt * WEIGHTS["equal"][CRITERIA]).sum(axis=1)
print("panel sizes:", summary["n_tested"].describe()[["min", "50%", "max"]].to_dict())
pd.DataFrame({"count (main)": wqml.agreement(t, "equal"),
              "per tested parameter": wqml.agreement(t.assign(pt=score_pt), "pt")}).T

## 5b. Common panel and the experts' own panel bias
Two checks of the panel-size effect:
1. Recompute every criterion from only the parameters tested almost everywhere (BOD at 32 of 33 sources, pH at 29, chromium at 28).
2. Ask whether the *experts' scores* themselves follow the number of tested parameters (within-area Spearman of `n_tested` against the paper's score; negative = more tests, scored as more polluting).

In [ ]:
from scipy.stats import spearmanr
rows = {"all parameters (main)": wqml.agreement(t, "equal")}
for panel in [["BOD", "pH", "Cr"], ["BOD", "Cr"], ["BOD"]]:
    s_p = wqml.summarise(df[df["pollutant"].isin(panel)])
    sc = (wqml.local_weights(s_p) * WEIGHTS["equal"][CRITERIA]).sum(axis=1)
    rows["common panel: " + ", ".join(panel)] = wqml.agreement(t.assign(cp=sc), "cp")
rows["panel size alone (fewer tests = cleaner)"] = wqml.agreement(t.assign(ps=-summary["n_tested"]), "ps")
display(pd.DataFrame(rows).T.round(3))
bias = {a: spearmanr(g["n_tested"], g["fuzzy_weight"]).statistic
        for a, g in summary[["industrial_area", "n_tested"]].join(labels).groupby("industrial_area")}
pd.Series(bias, name="experts' score vs n_tested (Spearman)").round(2)

## 6. Standard indices (exact definitions)
For comparison only. All use outlet concentrations against CPCB limits, the worst season per parameter, and are oriented so lower = more polluting.
- **Nemerow**: √((mean(Pᵢ)² + max(Pᵢ)²)/2), Pᵢ = C/limit over tested parameters (pH excluded).
- **HPI** (metals: Cr, Cr(VI), Cu, Ni, Pb, Cd, Zn, As, Fe, Mn): Σ Wᵢ Qᵢ / Σ Wᵢ, Wᵢ = 1/limit, Qᵢ = 100·C/limit.
- **Excess × flow**: Σ toxicity × max(0, C/limit − 1) × flow (MLD); needs flow.
- **TOPSIS** on the criteria matrix of section 2 with CRITIC weights.
- **CCME WQI** is left out: it needs the same parameters for every source, and panels here differ.

Higher agreement with the experts does not make an index a better assessment.

In [ ]:
o = df.dropna(subset=["outlet", "limit"]).copy()
o["P"] = o["outlet"] / o["limit"]
worst_p = o.groupby(["source_id", "pollutant"])["P"].max()
nemerow = worst_p.groupby(level=0).apply(lambda p: np.sqrt((p.mean() ** 2 + p.max() ** 2) / 2))
metals = worst_p[worst_p.index.get_level_values(1).isin(["Cr", "Cr6", "Cu", "Ni", "Pb", "Cd", "Zn", "As", "Fe", "Mn"])].reset_index()
metals["W"] = 1 / metals["pollutant"].map(wqml.LIMITS)
hpi = metals.groupby("source_id").apply(lambda m: (m["W"] * 100 * m["P"]).sum() / m["W"].sum())
excess = worst_p.reset_index().assign(x=lambda d: (d["P"] - 1).clip(lower=0) * d["pollutant"].map(wqml.TOXICITY).fillna(1))
excess_flow = excess.groupby("source_id")["x"].sum() * summary["C1"]
Wc = WEIGHTS["CRITIC"][used]
V = G / np.sqrt((G ** 2).sum()) * Wc
topsis = np.sqrt(((V - V.min()) ** 2).sum(axis=1)) / (np.sqrt(((V - V.max()) ** 2).sum(axis=1)) + np.sqrt(((V - V.min()) ** 2).sum(axis=1)))
INDICES = {"Nemerow": -nemerow, "HPI (metals)": -hpi, "excess x flow": -excess_flow, "TOPSIS-CRITIC": topsis}
rows = {}
for name, v in INDICES.items():
    ti = t[["industrial_area", "fuzzy_weight"]].assign(S=v).dropna(subset=["S"])
    rows[name] = wqml.agreement(ti, "S").to_dict() | {"industries": len(ti)}
rows["B, equal weights"] = wqml.agreement(t, "equal").to_dict() | {"industries": len(t)}
pd.DataFrame(rows).T.round(3)

## Before using on real Unnao data
- [ ] Confirm the INFERRED Magarwara station mapping in `data/uppcb/stations.csv` (11 of 33 labels)
- [ ] `wqml.C5_CLASS`, `wqml.TOXICITY`, `wqml.PH_RANGE` confirmed by domain experts
- [ ] Every industry where B and the paper disagree reviewed with `explain()`
- [ ] C6–C9 need upstream/downstream and paired wells; until then they are mostly neutral